In [1]:
import numpy as np
from pyscf import gto, sftda
from pyscf.sftda.uhf_sf import get_ab_sf
import sys
sys.path.append('/home/peter/Research/ZeroExcitationEnergy/code/src')
from tools import extract_state

In [2]:
basis = 'cc-pvtz'
fun = 'B3LYP'

atom = '''
O           0.57758       -0.13820       -0.00000
N          -0.56017        0.27787        0.00000
H          -1.26308       -0.50093        0.00000
'''
mol = gto.M(atom=atom, charge=0, spin=2, basis=basis)

mf = mol.UKS(xc=fun)
mf.kernel()
gse = mf.e_tot

a, b = get_ab_sf(mf, collinear_samples=-1)
A_baba, A_abab = a
B_baab, B_abba = b

n_occ_a, n_virt_b = A_abab.shape[0], A_abab.shape[1]
n_occ_b, n_virt_a = B_abba.shape[2], B_abba.shape[3]

A_abab_2d = A_abab.reshape((n_occ_a*n_virt_b,n_occ_a*n_virt_b), order='C')
B_abba_2d = B_abba.reshape((n_occ_a*n_virt_b,n_occ_b*n_virt_a), order='C')
B_baab_2d = B_baab.reshape((n_occ_b*n_virt_a,n_occ_a*n_virt_b), order='C')
A_baba_2d = A_baba.reshape((n_occ_b*n_virt_a,n_occ_b*n_virt_a), order='C')

Casida_matrix = np.block([[ A_abab_2d, B_abba_2d],
                        [-B_baab_2d,-A_baba_2d]])

eigenvals, eigenvecs = np.linalg.eig(Casida_matrix)

idxt = eigenvals.real.argsort()
eigenvals = eigenvals[idxt]
eigenvecs = eigenvecs[:, idxt]

norms = np.linalg.norm(eigenvecs[:n_occ_a * n_virt_b], axis=0)**2
norms -= np.linalg.norm(eigenvecs[n_occ_a * n_virt_b:], axis=0)**2

sfd_eigenvals = eigenvals[norms>0]
sfd_eigenvecs = eigenvecs[:, norms>0].transpose(1,0)

def norm_xy(z):
    x = z[:n_occ_a*n_virt_b].reshape(n_occ_a,n_virt_b)
    y = z[n_occ_a*n_virt_b:].reshape(n_occ_b,n_virt_a)
    norm = np.linalg.norm(x)**2 - np.linalg.norm(y)**2
    norm = np.sqrt(1./norm)
    return ((0,x*norm), (y*norm,0))

mftd1 = sftda.TDA_SF(mf)
mftd1.nstates = len(sfd_eigenvals)
mftd1.extype = 1
mftd1.collinear_samples = -1
mftd1.e = sfd_eigenvals
mftd1.xy = [norm_xy(z) for z in sfd_eigenvecs]

S = extract_state(mf, mftd1)[0]

/home/peter/Research/SFTDApplication/CollinearSF/RSH/pyscf/dft/libxc.py:1102: UserWarning: Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, corresponding to the original definition by Stephens et al. (issue 1480) and the same as the B3LYP functional in Gaussian. To restore the VWN5 definition, you can put the setting "B3LYP_WITH_VWN5 = True" in pyscf_conf.py
  warnings.warn('Since PySCF-2.3, B3LYP (and B3P86) are changed to the VWN-RPA variant, '


converged SCF energy = -130.505688381798  <S^2> = 2.0072359  2S+1 = 3.0048201
0-th state with excitation energy = 0.0005, S^2 = 0.0074 Norms: 0.995@(9, 1), 0.001@(9, 3)
2-th state with excitation energy = 1.8951, S^2 = 0.0155 Norms: 0.567@(9, 2), 0.430@(8, 1)
3-th state with excitation energy = 4.0917, S^2 = 0.0108 Norms: 0.997@(8, 2), 0.001@(9, 1)


In [4]:
extract_state(mf, mftd1, Smin=1.9, Smax=2.7)

1-th state with excitation energy = 1.6413, S^2 = 2.0037 Norms: 0.565@(8, 1), 0.431@(9, 2)
14-th state with excitation energy = 12.9006, S^2 = 1.9948 Norms: 0.991@(7, 3), 0.002@(7, 1)
15-th state with excitation energy = 13.2245, S^2 = 1.9829 Norms: 0.978@(6, 3), 0.014@(9, 5)
18-th state with excitation energy = 14.8987, S^2 = 1.9185 Norms: 0.917@(5, 3), 0.069@(4, 1)


[1, 14, 15, 18]